# Image Captioning

In [ ]:
# !pip install nltk pandas Pillow wandb numpy

In [ ]:
import os
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.utils.data
import wandb
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from PIL import Image
from tqdm.auto import tqdm

from dataloaders.dataset import Flickr8kDataset
from models.baseline import Baseline
from utils.vocab import GenerationVocab

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Configuration

In [ ]:
config = dict(
    epochs=15,
    batch_size=32,
    lr=2e-4,
    hidden_dim=256,
    char_dim=512,
    num_layers=2,
    dropout=0.3,
    weight_decay=1e-4,
    label_smoothing=0.1,
    scheduler_factor=0.5,
    scheduler_patience=2,
    early_stop_patience=4,
    max_len=30,
    teacher_forcing=0.5,
    data_dir="data/flickr8k",
    checkpoint_dir="checkpoints",
    min_word_freq=2,
    num_workers=2,
)



## Vocabulary

In [ ]:
def build_vocab(captions_file, min_freq=2):
    df = pd.read_csv(captions_file)
    df.columns = ["image", "caption"]
    unique_images = sorted(df["image"].unique())
    train_images = set(unique_images[:int(len(unique_images) * 0.75)])
    train_df = df[df["image"].isin(train_images)]
    word_counts = Counter(
        word
        for caption in train_df["caption"].str.lower()
        for word in caption.split()
    )
    words = sorted(w for w, c in word_counts.items() if c >= min_freq)
    return GenerationVocab(words)


captions_file = os.path.join(config["data_dir"], "captions.txt")
vocab = build_vocab(captions_file, config["min_word_freq"])
print(f"Vocabulary size: {len(vocab)}")

## Datasets & Dataloaders

In [ ]:
common_ds_kwargs = dict(
    image_dir=os.path.join(config["data_dir"], "Images"),
    captions_file=captions_file,
    vocab=vocab,
    max_length=config["max_len"],
)
loader_kwargs = dict(
    batch_size=config["batch_size"],
    num_workers=config["num_workers"],
    pin_memory=True,
)

train_dataset = Flickr8kDataset(**common_ds_kwargs, split="train")
val_dataset   = Flickr8kDataset(**common_ds_kwargs, split="val")

train_loader = torch.utils.data.DataLoader(train_dataset, shuffle=True,  **loader_kwargs)
val_loader   = torch.utils.data.DataLoader(val_dataset,   shuffle=False, **loader_kwargs)

print(f"Train: {len(train_dataset)} samples | Val: {len(val_dataset)} samples")

In [ ]:
# Sanity check — inspect one sample
image, token_ids, image_name = train_dataset[0]
print(f"Image name  : {image_name}")
print(f"Image shape : {image.shape}")
print(f"Token IDs   : {token_ids.tolist()}")
print(f"Decoded     : {vocab.unprepare(token_ids.tolist())}")

## Model, Loss & Optimizer

In [ ]:
model = Baseline(
    num_char=len(vocab),
    hidden_dimension=config["hidden_dim"],
    char_dimension=config["char_dim"],
    text_max_len=config["max_len"],
    num_layers=config["num_layers"],
    dropout=config["dropout"],
).to(device)

pad_idx   = vocab.token2index[vocab.PAD_TAG]
criterion = nn.CrossEntropyLoss(
    ignore_index=pad_idx, label_smoothing=config["label_smoothing"]
)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_params, lr=config["lr"], weight_decay=config["weight_decay"]
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=config["scheduler_factor"],
    patience=config["scheduler_patience"],
)

n_params       = sum(p.numel() for p in model.parameters())
n_trainable    = sum(p.numel() for p in trainable_params)
print(f"Parameters: {n_params:,} (trainable: {n_trainable:,})")

## Training Utilities

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, teacher_forcing):
    model.train()
    total_loss = 0.0
    for images, token_ids, _ in tqdm(loader, leave=False, desc="Train"):
        images, token_ids = images.to(device), token_ids.to(device)
        outputs = model(images, token_ids, teacher_forcing=teacher_forcing)
        loss = criterion(
            outputs[:, :-1].reshape(-1, outputs.size(-1)),
            token_ids[:, 1:].reshape(-1),
        )
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def _decode(token_ids, vocab, stop_idx, pad_idx):
    skip = {vocab.token2index[vocab.START_TAG], vocab.token2index[vocab.UNKNOWN_TAG]}
    words = []
    for tok in token_ids:
        if tok in (stop_idx, pad_idx):
            break
        if tok not in skip:
            words.append(vocab.index2token.get(tok, ""))
    return words


def _generate(model, images, vocab, device):
    """Autoregressive generation for a batch of images."""
    pad_idx = vocab.token2index[vocab.PAD_TAG]
    unk_idx = vocab.token2index[vocab.UNKNOWN_TAG]
    max_len = model.text_max_len
    dummy = torch.full((images.size(0), max_len), pad_idx, device=device)
    dummy[:, 0] = vocab.token2index[vocab.START_TAG]
    with torch.no_grad():
        logits = model(images, dummy, teacher_forcing=0.0)
        logits[..., unk_idx] = float("-inf")
        return logits.argmax(dim=-1)


def evaluate(model, loader, criterion, device, vocab):
    model.eval()
    total_loss = 0.0
    refs_by_image, hyp_by_image = {}, {}
    stop_idx = vocab.token2index[vocab.STOP_TAG]
    pad_idx  = vocab.token2index[vocab.PAD_TAG]

    with torch.no_grad():
        for images, token_ids, image_names in tqdm(loader, leave=False, desc="Val"):
            images, token_ids = images.to(device), token_ids.to(device)
            outputs = model(images, token_ids)
            loss = criterion(
                outputs[:, :-1].reshape(-1, outputs.size(-1)),
                token_ids[:, 1:].reshape(-1),
            )
            total_loss += loss.item()

            pred_ids = _generate(model, images, vocab, device).cpu().tolist()
            for i, name in enumerate(image_names):
                ref = _decode(token_ids[i].cpu().tolist(), vocab, stop_idx, pad_idx)
                refs_by_image.setdefault(name, []).append(ref)
                if name not in hyp_by_image:
                    hyp_by_image[name] = _decode(pred_ids[i], vocab, stop_idx, pad_idx)

    names = list(hyp_by_image.keys())
    references = [refs_by_image[n] for n in names]
    hypotheses = [hyp_by_image[n] for n in names]

    smoother = SmoothingFunction().method1
    bleu = corpus_bleu(references, hypotheses, smoothing_function=smoother)
    return total_loss / len(loader), bleu


def bleu_subset(model, loader, vocab, device, n_batches=10):
    model.eval()
    refs_by_image, hyp_by_image = {}, {}
    stop_idx = vocab.token2index[vocab.STOP_TAG]
    pad_idx  = vocab.token2index[vocab.PAD_TAG]

    with torch.no_grad():
        for i, (images, token_ids, image_names) in enumerate(loader):
            if i >= n_batches:
                break
            images = images.to(device)
            pred_ids = _generate(model, images, vocab, device).cpu().tolist()
            for j, name in enumerate(image_names):
                ref = _decode(token_ids[j].tolist(), vocab, stop_idx, pad_idx)
                refs_by_image.setdefault(name, []).append(ref)
                if name not in hyp_by_image:
                    hyp_by_image[name] = _decode(pred_ids[j], vocab, stop_idx, pad_idx)

    names = list(hyp_by_image.keys())
    references = [refs_by_image[n] for n in names]
    hypotheses = [hyp_by_image[n] for n in names]

    smoother = SmoothingFunction().method1
    return corpus_bleu(references, hypotheses, smoothing_function=smoother)


def save_checkpoint(model, optimizer, scheduler, epoch, val_bleu, path):
    checkpoint_dir = os.path.dirname(path)
    if checkpoint_dir:
        os.makedirs(checkpoint_dir, exist_ok=True)
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "val_bleu": val_bleu,
    }, path)


def load_checkpoint(path, model, optimizer, scheduler=None):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if "val_bleu" in ckpt:
        return ckpt["epoch"], ckpt["val_bleu"]
    return ckpt["epoch"], float("-inf")

## Resume from Checkpoint (optional)

In [ ]:
RESUME = None  # e.g. "checkpoints/best_model.pt"

start_epoch              = 0
best_val_bleu            = float("-inf")
epochs_since_improvement = 0
in_sweep                 = False  # Notebook runs are always standalone.

if RESUME:
    start_epoch, best_val_bleu = load_checkpoint(RESUME, model, optimizer, scheduler)
    start_epoch += 1
    print(f"Resumed from epoch {start_epoch - 1}, best_val_bleu={best_val_bleu:.4f}")

## Weights & Biases

In [ ]:
wandb.login()

wandb.init(
    project="image-captioning",
    config=config,
    resume="allow" if RESUME else None,
)
wandb.watch(model, log="gradients", log_freq=100)

## Training Loop

In [ ]:
checkpoint_path = os.path.join(config["checkpoint_dir"], "best_model.pt")

for epoch in tqdm(range(start_epoch, config["epochs"]), desc="Epochs"):
    train_loss               = train_epoch(model, train_loader, criterion, optimizer, device, config["teacher_forcing"])
    val_loss, val_bleu       = evaluate(model, val_loader, criterion, device, vocab)
    train_bleu               = bleu_subset(model, train_loader, vocab, device, n_batches=10)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    wandb.log({
        "epoch":      epoch,
        "train/loss": train_loss,
        "val/loss":   val_loss,
        "train/bleu": train_bleu,
        "val/bleu":   val_bleu,
        "lr":         current_lr,
    })
    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
        f"train_bleu={train_bleu:.4f}  val_bleu={val_bleu:.4f}  "
        f"lr={current_lr:.2e}"
    )

    if val_bleu > best_val_bleu:
        best_val_bleu            = val_bleu
        epochs_since_improvement = 0
        save_checkpoint(model, optimizer, scheduler, epoch, val_bleu, checkpoint_path)
        print(f"  -> Checkpoint saved (epoch {epoch}, val_bleu={val_bleu:.4f})")
    elif not in_sweep:
        epochs_since_improvement += 1
        if epochs_since_improvement >= config["early_stop_patience"]:
            print(f"Early stopping: no val/bleu improvement in {config['early_stop_patience']} epochs.")
            break

wandb.finish()

## Sample Caption Generation

In [ ]:
def show_samples(model, dataset, vocab, device, n=4):
    model.eval()
    stop_idx = vocab.token2index[vocab.STOP_TAG]
    pad_idx  = vocab.token2index[vocab.PAD_TAG]

    indices = random.sample(range(len(dataset)), n)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))

    for ax, idx in zip(axes, indices):
        image_name, _ = dataset.data[idx]
        image_tensor, _, _ = dataset[idx]

        pred_ids = _generate(model, image_tensor.unsqueeze(0).to(device), vocab, device)
        words    = _decode(pred_ids[0].cpu().tolist(), vocab, stop_idx, pad_idx)

        ax.imshow(Image.open(os.path.join(dataset.image_dir, image_name)))
        ax.set_title(" ".join(words), fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_samples(model, val_dataset, vocab, device)